# import necessary libraries

In [1]:
from openai import OpenAI
import pandas as pd
from retriv import SparseRetriever, DenseRetriever
from typing import List, Literal
import warnings
from typing import Dict
from transformers import AutoTokenizer
import os
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm


/home/rfzhang/anaconda3/envs/vllm2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# information about the sota models and tasks

In [ ]:
models_info = {
    'deepseek-v3': {
        'api-key': 'xxxxx', # replace with your own API key
        'base-url': 'https://api.deepseek.com',
        'tokenizer': '/data/rfzhang/harmfultext/model/deepseek-v3-tokenizer'
    }
}
system_prompt = """#### Positioning
- Assistant Name: Harmful Text Detector
- Primary Task: For a given text, based on the provided definitions in Knowledge Base strictly, identify all applicable labels among 'benign', 'negative', 'toxic', and 'spam', and explain the reasoning. Also, determine the **most likely** single label among them for multi-class classification tasks.

#### Capabilities
- Text Analysis: Accurately analyze the content and characteristics of text.
- Multi-Label Classification: Determine all relevant labels for the text (e.g., a text can be both 'negative' and 'toxic').
- Multi-Class Determination: Identify the single label that best represents the text if only one label must be chosen.

#### Knowledge Base
- Message Labels:
  - 'benign': The text is neutral, factual, or positive in tone. It does not contain emotional harm, toxicity, manipulation, misleading implications, or commercial spam. This includes objective descriptions, factual reports (e.g., news), and informative content that respects communication norms.
  - 'negative': emotionally unpleasant, critical, or pessimistic in tone, but not clearly abusive or violating.
  - 'toxic': offensive, hateful, bullying, threatening, or severely harmful content.
  - 'spam': irrelevant advertising, repetitive promotions, low-effort engagement bait.

#### Instructions for Use
- Input: A piece of text.
- Output: A JSON object containing:
  - 'reason': a concise explanation of why these labels were chosen, based on tone, content, and language use
  - 'labels': a list of all applicable labels from ['benign', 'negative', 'toxic', 'spam']
  - 'label': the **single** most likely label among them
  
#### Example Output
{
  "reason": "The text expresses strong personal attacks and uses offensive language, which is clearly toxic. Its hostile tone also conveys negativity.",
  "labels": ["negative", "toxic"],
  "label": "toxic"
}"""




# use deepseek to predict text's label and reason

In [ ]:
# read data
test_df = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted.csv')

model_info = models_info['deepseek-v3']
outputs = []
client = OpenAI(api_key=model_info['api-key'] ,base_url=model_info['base-url'])
for text in tqdm(test_df['text'], desc="Predicting"):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages, # type: ignore
        logprobs=False,
        max_tokens=500, # prevent truncation
        temperature=0, 
        top_p=1.0, 
        response_format={
        'type': 'json_object'
        },
        # stop=labels,
    )
    outputs.append(response)


In [ ]:
reasons = []
labelses = []
labels = []

for output in outputs:
    try:
        json_output = json.loads(output.choices[0].message.content)
    except:
        json_output = {'reason': '','labels': [], 'label': 'benign'}
    reasons.append(json_output['reason'])
    #不同标签之间用, 链接
    labelses.append(', '.join(json_output['labels']))
    labels.append(json_output['label'])
print(reasons)
print(labelses)
print(labels)
test_df['reason'] = reasons + ['']*(len(test_df) - len(reasons))
test_df['labelses'] = labelses + ['']*(len(test_df) - len(labelses))
test_df['label'] = labels + ['']*(len(test_df) - len(labels))
test_df.to_csv('../datasets/mastodon2025-en-shuffled-predicted.csv', index=False)
    

['The text is positive in tone and does not contain any elements of emotional harm, toxicity, manipulation, misleading implications, or commercial spam. It is a friendly and supportive message.', "The text is a neutral, factual report about a public figure's accusation, which does not contain emotional harm, toxicity, manipulation, misleading implications, or commercial spam. It is informative and respects communication norms.", "The text is promoting a new game on Steam with a link and price, which fits the definition of 'spam' as it is irrelevant advertising and promotional content. It does not contain any negative, toxic, or benign characteristics as per the provided definitions.", 'The text is positive in tone and does not contain any elements of emotional harm, toxicity, manipulation, misleading implications, or commercial spam.', 'The text is a neutral, factual report about a Debian package being marked for acceptance to a queue, which does not contain emotional harm, toxicity, m

In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed


# read data
test_df = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted.csv')

# if there are no reason, labelses, or labels columns, add them
for col in ['reason', 'labelses', 'label']:
    if col not in test_df.columns:
        test_df[col] = ''

# initialize client
model_info = models_info['deepseek-v3']
client = OpenAI(api_key=model_info['api-key'], base_url=model_info['base-url'])

# predict function
def predict_single(index, text):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            logprobs=False,
            max_tokens=500,
            temperature=0,
            top_p=1.0,
            response_format={'type': 'json_object'}
        )
        output = json.loads(response.choices[0].message.content)
        reason = output.get('reason', '')
        labels_str = ', '.join(output.get('labels', []))
        label = output.get('label', 'benign')
    except Exception:
        reason, labels_str, label = '', '', 'benign'
    return index, reason, labels_str, label

# get the unpredicted rows
unpredicted = test_df[test_df['reason'].isna() | (test_df['reason'] == '')]

# show the number of unpredicted rows
print(f"🧠 unpredicted rows: {len(unpredicted)}")

# use multithreading to predict the unpredicted rows
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(predict_single, idx, row['text']) for idx, row in unpredicted.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Predicting"):
        index, reason, labels_str, label = future.result()
        test_df.loc[index, 'reason'] = reason
        test_df.loc[index, 'labelses'] = labels_str
        test_df.loc[index, 'label'] = label

# save the result
test_df.to_csv('../datasets/mastodon2025-en-shuffled-predicted.csv', index=False)
print("✅ Predicted and saved")


🧠 待预测样本数: 3884


Predicting: 100%|██████████| 3884/3884 [1:52:42<00:00,  1.74s/it]  

✅ 所有预测完成并已保存！


# use deepseek to predict text's reason

In [3]:
system_prompt = """#### Positioning
- Assistant Name: Harmful Text Detector
- Primary Task: Given a piece of text and its associated labels (multi-label and multi-class), provide a concise and well-justified explanation (reason) for why those labels were assigned, strictly according to the definitions in the Knowledge Base.

#### Capabilities
- Text Reasoning: Analyze the tone, content, and intent of the input text.
- Justification Generation: Generate reasoning that logically connects the label assignments to the definitions and textual evidence.

#### Knowledge Base
- Message Labels:
  - 'benign': The text is neutral, factual, or positive in tone. It does not contain emotional harm, toxicity, manipulation, misleading implications, or commercial spam. This includes objective descriptions, factual reports (e.g., news), and informative content that respects communication norms.
  - 'harmful': Harmful text refers to content that may cause discomfort, harm, or negative effects on others. It includes:
    - 'negative': emotionally unpleasant, critical, or pessimistic in tone, but not clearly abusive or violating.
    - 'toxic': offensive, hateful, bullying, threatening, or severely harmful content.
    - 'spam': irrelevant advertising, repetitive promotions, low-effort engagement bait.

#### Instructions for Use
- Input: A text string, a list of assigned labels (multi-label), and a single most likely label (multi-class).
- Output: A JSON object containing:
  - 'reason': a concise explanation that supports the chosen labels, grounded in the tone, language, and content of the text, and aligned with the definitions in the Knowledge Base.

#### Example Input
text = You're such a useless waste of space. No one wants you here.
labels = ["negative", "toxic"]
label = toxic

#### Example Output
{
  "reason": "The text contains direct personal attacks and demeaning language, which makes it toxic. Its harsh and emotionally hostile tone also supports the negative label. Toxicity is the most prominent feature."
}
"""

In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed

file = '../datasets/mastodon2025-en-shuffled-predicted-harmful.csv'
# read data
test_df = pd.read_csv(file)

# if columns not exist, add them
for col in ['t-reason-deepseek']:
    if col not in test_df.columns:
        test_df[col] = ''

# initiate model
model_info = models_info['deepseek-v3']
client = OpenAI(api_key=model_info['api-key'], base_url=model_info['base-url'])

# predict single sample
def predict_single(index, text, label, labels):
    input = f"text = {text}\nlabels = {labels}\nlabel = {label}"
    # print(input)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": input},
    ]
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            logprobs=False,
            max_tokens=500,
            temperature=0,
            top_p=1.0,
            response_format={'type': 'json_object'}
        )
        output = json.loads(response.choices[0].message.content)
        reason = output.get('reason', '')
    except Exception:
        reason = ''
    return index, reason

# get the tail 500 samples of the test_df
test_df_tail = test_df.tail(500)
unpredicted = test_df_tail[test_df_tail['t-reason-deepseek'].isna() | (test_df_tail['t-reason-deepseek'] == '')]
# show the length of the unpredicted
print(f"🧠 unpredicted: {len(unpredicted)}")

# use ThreadPoolExecutor to predict
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(predict_single, idx, row['text'], row['label'], row['labels']) for idx, row in unpredicted.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Predicting"):
        index, reason = future.result()
        test_df.loc[index, 't-reason-deepseek'] = reason

# 保存结果
test_df.to_csv(file, index=False)
print("✅ Predicted and saved!")


🧠 待预测样本数: 500


Predicting: 100%|██████████| 500/500 [11:18<00:00,  1.36s/it]

✅ 所有预测完成并已保存！


# use deepseek to give the definition of the label

In [ ]:
new_shots = 48
test_df_predicted_benign_1500 = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted-benign.csv').head(1500)
test_df_predicted_harmful_1500 = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted-harmful.csv').head(1500)
test_df = pd.concat([test_df_predicted_benign_1500, test_df_predicted_harmful_1500], ignore_index=True)
test_df.rename(columns={'label': 'original_label'}, inplace=True)
test_df['label'] = test_df['original_label'].apply(lambda x: 'benign' if x == 'benign' else 'harmful')
# print(test_df.groupby('predicted_label').size())
# test_df = test_df.head(10)
# test_df.to_csv('./demos.csv', index=False)
#get the tail 500 samples of the test_df
new_shots_df_predicted_benign_500 = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted-benign.csv').tail(500)
new_shots_df_predicted_harmful_500 = pd.read_csv('../datasets/mastodon2025-en-shuffled-predicted-harmful.csv').tail(500)
new_shots_df = pd.concat([new_shots_df_predicted_benign_500, new_shots_df_predicted_harmful_500], ignore_index=True)
new_shots_df.rename(columns={'label': 'original_label'}, inplace=True)
new_shots_df['label'] = new_shots_df['original_label'].apply(lambda x: 'benign' if x == 'benign' else 'harmful')

sample_sizes = {
    'benign': new_shots // 2,
    'negative': new_shots // 6,
    'spam': new_shots // 6,
    'toxic': new_shots // 6
}
df = pd.concat([df_group.sample(n=sample_sizes[label], random_state=42) for label, df_group in new_shots_df.groupby('original_label') if label in sample_sizes])
# display(df)

In [9]:
system_prompt = """#### Positioning\n- Assistant Name: Harmful Text Detector\n- Primary Task: Classify whether a given text is harmful or benign.\n\n#### Capabilities\n- Text Analysis: Accurately analyze the content and characteristics of text.\n- Harmfulness Classification: Determine whether a text is 'harmful' or 'benign'.\n\n#### Knowledge Base\n- Message Categories and Definitions:\n - harmful: Harmful text refers to content that may cause discomfort, harm, or negative effects on others. It includes:\n  - spam: Unsolicited, often irrelevant messages\n  - toxic: Hostile, offensive language\n  - negative emotion text: Messages expressing sadness, anger, or frustration that may affect others negatively\n - benign: Benign text does not contain harmful content. It includes:\n  - ham: Non-spam, relevant messages\n  - non-toxic text: Neutral or friendly language\n  - positive emotion text: Messages expressing happiness, gratitude, or optimism\n\n#### Instructions for Use\n- Input: A piece of text.\n- Output: Only output the category of the text ('harmful' or 'benign'), without any additional explanation.
===
This is the prompt for me to use a large model for ICL to achieve text classification. I think my category definition is not very accurate. Next, I will provide you with a certain number of examples of the corresponding categories. Please adjust the category definition based on the examples. The definition is required to be concise and reasonable."""

def generate_examples(df, label):
    texts_list = df[df['original_label'] == label]['text'].tolist()
    examples = "\n".join([f"Example {index+1}:\n{text}" for index, text in enumerate(texts_list)])
    return f"The examples of {label}:\n===\n{examples}\n==="

message = generate_examples(df, 'benign')
message += "\n" + generate_examples(df, 'negative')
message += "\n" + generate_examples(df, 'spam')
message += "\n" + generate_examples(df, 'toxic')
message += "\nPlease adjust the category definition based on the examples. The definition is required to be concise and reasonable."
display(message)

'The examples of benign:\n===\nExample 1:\n#HouseEthics #Committee #Report on #MattGaetz https://ethics.house.gov/wp-content/uploads/2024/12/Committee-Report.pdf #OnlyTheBestPeople #Trump2.0 #BlackMastodon\nExample 2:\n“The Open Source Torment Nexus”https://tante.cc/2025/02/07/the-open-source-torment-nexus/\nExample 3:\n#SSH #Cybersecurity #Cyberspies #Hackinggrouphttps://flip.it/aKxH8r\nExample 4:\n#Evil #Netflix Season 3 of Evil will be added to Netflix on December 31st. Season 4 will remain a Paramount+ exclusive. \nA New Season Of Evil Is Finally Being Added To Netflix Later This Month\n"Evil was canceled by Paramount+."https://screenrant.com/evil-season-3-netflix-release-date/#:~:text=That%27s%20all%20about%20to%20change,exclusive%20for%20the%20time%2Dbeing.\nExample 5:\nYour browser is not supported https://www.byteseu.com/?p=578513 #Sports\nExample 6:\nGonna get more active here because Bluesky seems to be already imploding :blobdab:\nExample 7:\nAt least the shareholders are ha

In [12]:
# read data

model_info = models_info['deepseek-v3']
client = OpenAI(api_key=model_info['api-key'] ,base_url=model_info['base-url'])
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": message},
]
    
response = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages, # type: ignore
        logprobs=False,
        max_tokens=8192, # prevent truncation
        temperature=0, 
        top_p=1.0, 
        # stop=labels,
    )
print(response.choices[0].message.content)


Here’s the revised category definition based on the examples:

### Message Categories and Definitions:
- **harmful**: Content that is disruptive, offensive, or manipulative. Includes:
  - **spam**: Unsolicited, repetitive, or promotional content (e.g., ads, NSFW links, crypto scams).
  - **toxic**: Hostile, inflammatory, or discriminatory language (e.g., personal attacks, bigotry, political vitriol).
  - **negative emotion text**: Expressions of frustration, despair, or cynicism that may spread negativity (e.g., "We're fucked," "Son of a bitch").

- **benign**: Neutral or constructive content. Includes:
  - **informative**: News, factual updates, or educational links.
  - **casual/conversational**: Everyday chatter, humor, or personal updates without harm.
  - **positive/neutral tone**: Gratitude, optimism, or neutral observations.

### Key Adjustments:
1. **Spam**: Explicitly added crypto scams and NSFW promotions based on examples.
2. **Toxic**: Broadened to include political vitriol